# GuardSLM — Complete Open-Source SLM Evaluation Suite (Google Colab)

This notebook contains pre-configured, ready-to-run execution blocks for **all free open-source SLMs** on the **40 matched pairs / 80 test cases** dataset:

1. **Qwen Guard** (`Qwen/Qwen2.5-1.5B-Instruct` / `Qwen/Qwen2.5-7B-Instruct`)
2. **Llama Guard 3** (`meta-llama/Llama-Guard-3-1B` / `meta-llama/Llama-Guard-3-8B`)
3. **WebGuard 7B** (`OSU-NLP/WebGuard-7B`)
4. **DynaGuard 8B** (`DynaGuard/DynaGuard-8B`)
5. **PolicyGuard 4B** (`PolicyGuard/PolicyGuard-4B`)
6. **Rule Baseline** (Deterministic State Rules)

### Step 1 — Clone / Update Repository

In [ ]:
import os
import sys

if os.path.exists('/content'):
    %cd /content
    if not os.path.exists('GuardSLM'):
        !git clone https://github.com/AkarshiAaryan/GuardSLM.git
    %cd GuardSLM
    !git pull

sys.path.insert(0, os.getcwd())
print(f"Project root directory: {os.getcwd()}")

### Step 2 — Install Free Open-Source Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers torch accelerate bitsandbytes

### Step 3 — Hardware & VRAM Check

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU. Enable GPU acceleration: Runtime -> Change runtime type -> T4 GPU.")

### Step 4 — Load & Validate Test Cases

In [ ]:
from src.data.validator import validate_dataset_file
from src.data.loader import load_test_cases

dataset_path = 'data/dummy/dummy_cases.json'
is_valid, errors = validate_dataset_file(dataset_path)

if is_valid:
    cases = load_test_cases(dataset_path)
    print(f"SUCCESS: Loaded {len(cases)} test cases across {len(set(c.pair_id for c in cases))} matched pairs!")
else:
    print("Validation errors:", errors)

### Step 5A — Model 1: Qwen Guard (`Qwen/Qwen2.5-1.5B-Instruct`)

In [ ]:
from src.models.qwen_guard import QwenGuardAdapter
from src.evaluation.runner import run_evaluation_for_model
from src.evaluation.metrics import calculate_overall_metrics
from src.evaluation.pair_metrics import calculate_pair_metrics

qwen_model = QwenGuardAdapter(name="qwen_guard", config={
    "enabled": True,
    "checkpoint": "Qwen/Qwen2.5-1.5B-Instruct",  # Or "Qwen/Qwen2.5-7B-Instruct"
    "load_in_4bit": False
})

qwen_results = run_evaluation_for_model(qwen_model, cases)
qwen_overall = calculate_overall_metrics(qwen_results)
qwen_pair = calculate_pair_metrics(qwen_results)

print(f"[Qwen Guard] Accuracy: {qwen_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {qwen_pair['pair_accuracy']*100:.1f}% | Flip Rate: {qwen_pair['context_flip_rate']*100:.1f}%")

### Step 5B — Model 2: Llama Guard 3 (`meta-llama/Llama-Guard-3-1B`)

In [ ]:
from src.models.llama_guard import LlamaGuardAdapter

# Note: If using Meta gated checkpoints, ensure you login first: huggingface_hub.login()
llama_model = LlamaGuardAdapter(name="llama_guard", config={
    "enabled": True,
    "checkpoint": "meta-llama/Llama-Guard-3-1B",
    "load_in_4bit": False
})

try:
    llama_results = run_evaluation_for_model(llama_model, cases)
    llama_overall = calculate_overall_metrics(llama_results)
    llama_pair = calculate_pair_metrics(llama_results)
    print(f"[Llama Guard 3] Accuracy: {llama_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {llama_pair['pair_accuracy']*100:.1f}%")
except Exception as e:
    print(f"Llama Guard note: {e}")

### Step 5C — Model 3: WebGuard 7B (`OSU-NLP/WebGuard-7B`)

In [ ]:
from src.models.webguard import WebGuardAdapter

webguard_model = WebGuardAdapter(name="webguard", config={
    "enabled": True,
    "checkpoint": "OSU-NLP/WebGuard-7B",
    "load_in_4bit": True  # 4-bit quantization for 7B parameter model
})

try:
    webguard_results = run_evaluation_for_model(webguard_model, cases)
    webguard_overall = calculate_overall_metrics(webguard_results)
    webguard_pair = calculate_pair_metrics(webguard_results)
    print(f"[WebGuard 7B] Accuracy: {webguard_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {webguard_pair['pair_accuracy']*100:.1f}%")
except Exception as e:
    print(f"WebGuard note: {e}")

### Step 5D — Model 4: DynaGuard 8B (`DynaGuard/DynaGuard-8B`)

In [ ]:
from src.models.dynaguard import DynaGuardAdapter

dynaguard_model = DynaGuardAdapter(name="dynaguard", config={
    "enabled": True,
    "checkpoint": "DynaGuard/DynaGuard-8B",
    "load_in_4bit": True
})

try:
    dynaguard_results = run_evaluation_for_model(dynaguard_model, cases)
    dynaguard_overall = calculate_overall_metrics(dynaguard_results)
    print(f"[DynaGuard 8B] Accuracy: {dynaguard_overall['overall_accuracy']*100:.1f}%")
except Exception as e:
    print(f"DynaGuard note: {e}")

### Step 5E — Model 5: PolicyGuard 4B (`PolicyGuard/PolicyGuard-4B`)

In [ ]:
from src.models.policyguard import PolicyGuardAdapter

policyguard_model = PolicyGuardAdapter(name="policyguard", config={
    "enabled": True,
    "checkpoint": "PolicyGuard/PolicyGuard-4B",
    "load_in_4bit": False
})

try:
    policyguard_results = run_evaluation_for_model(policyguard_model, cases)
    policyguard_overall = calculate_overall_metrics(policyguard_results)
    print(f"[PolicyGuard 4B] Accuracy: {policyguard_overall['overall_accuracy']*100:.1f}%")
except Exception as e:
    print(f"PolicyGuard note: {e}")

### Step 5F — Model 6: Deterministic Rule Baseline (`RuleBaseline`)

In [ ]:
from src.baselines.rule_baseline import RuleBaseline

rule_model = RuleBaseline(name="rule_baseline")
rule_results = run_evaluation_for_model(rule_model, cases)
rule_overall = calculate_overall_metrics(rule_results)
rule_pair = calculate_pair_metrics(rule_results)

print(f"[Rule Baseline] Accuracy: {rule_overall['overall_accuracy']*100:.1f}% | Pair Accuracy: {rule_pair['pair_accuracy']*100:.1f}%")

### Step 6 — Generate & Display Decision Gate Report

In [ ]:
from IPython.display import display, Markdown
from src.evaluation.failure_analysis import analyze_failures
from src.evaluation.decision_gate import generate_decision_gate_report

failures = analyze_failures(qwen_results)
report_path = 'reports/colab_decision_gate_report.md'
report_content = generate_decision_gate_report(qwen_overall, qwen_pair, failures, report_path)

print(f"Decision Gate Report generated at: {report_path}\n")
# Display formatted report directly inside notebook
display(Markdown(report_content))